# 10 (v2) - Log saved Phase 2 evidence to W&B as a documentation run (no model execution)

The earlier version logged the saved M8 files twice (runs f166xpj4 and i8giv0x1). This version:
- logs as `job_type="documentation"` with the tag `evidence-only`, so it is never confused with an inference run;
- computes a digest of the files first and **stops if an artifact with the same digest already exists** in the project, so old results are not uploaded again as if new;
- records the git commit of the evidence files and keeps T4 (historical benchmark) and CPU (31 Oct regression) values under separate keys.

No model is run and no value is changed. The W&B key is read from Colab Secrets (`WANDB_API_KEY`) or a hidden prompt.


In [ ]:
# 1. Evidence files at a pinned commit
EVIDENCE_REV = "phase2-submission-hardening"   # replace with the merge commit SHA once merged
!git clone -q https://github.com/Yasser-20-alhajlah/smart-pet-vision /content/spv-repo 2>/dev/null || true
!git -C /content/spv-repo fetch -q origin {EVIDENCE_REV} && git -C /content/spv-repo checkout -q FETCH_HEAD
!pip -q install wandb==0.28.1
import json, csv, hashlib, subprocess, pathlib
SHA = subprocess.run(["git", "-C", "/content/spv-repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
EV = pathlib.Path("/content/spv-repo/smart-pet-vision/phase2/report/evidence")
files = sorted(p for p in EV.iterdir() if p.name.startswith(("M8_", "P2_")) and p.suffix in {".json", ".csv"})
if not files: raise FileNotFoundError(f"No evidence files in {EV}")
digest = hashlib.sha256(b"".join(p.name.encode() + hashlib.sha256(p.read_bytes()).digest() for p in files)).hexdigest()
print("commit", SHA, "| files", len(files), "| digest", digest[:16])


In [ ]:
# 2. Stop if this exact set of files was already logged
import wandb
try:
    from google.colab import userdata; key = userdata.get("WANDB_API_KEY")
except Exception:
    from getpass import getpass; key = getpass("W&B API key (hidden): ")
wandb.login(key=key, relogin=True, verify=True); del key
ENTITY, PROJECT, ART = "yasser-alhajlah-kaust", "smart-pet-vision", "phase2-evidence-documentation"
api = wandb.Api()
existing = []
try:
    for v in api.artifacts(f"{ENTITY}/{PROJECT}/{ART}", type_name="evidence"):
        if v.metadata.get("files_digest") == digest: existing.append(v.name)
except Exception as e:
    print("No earlier documentation artifact found or not readable:", type(e).__name__)
if existing:
    raise SystemExit(f"Already logged as {existing}; nothing uploaded.")


In [ ]:
# 3. One documentation run; T4 and CPU values under separate keys
pt = json.loads((EV / "M8_benchmark_pytorch.json").read_text())
cpu = json.loads((EV / "P2_cpu_http_validation_mu2tolkg.json").read_text())
with wandb.init(entity=ENTITY, project=PROJECT, name=f"evidence-doc-{SHA[:7]}", job_type="documentation",
                tags=["phase2", "evidence-only", "no-model-run"], config={"evidence_commit": SHA, "files_digest": digest},
                notes="Documentation of saved evidence files. No model was executed; values are copies of saved files.") as run:
    run.summary.update({f"t4_historical/pytorch_latency_{k}_ms": v for k, v in pt["latency_ms"].items()})
    run.summary.update({f"cpu_31oct/http_latency_{k}_ms": v for k, v in cpu["http_latency_ms"].items()})
    run.summary["source_runs"] = "T4: fqow5ly7 and M8_benchmark_*.json; CPU: f8manl0i, mu2tolkg"
    art = wandb.Artifact(ART, type="evidence", metadata={"files_digest": digest, "evidence_commit": SHA},
                         description="Saved Phase 2 evidence files (unchanged); documentation only.")
    for p in files: art.add_file(str(p))
    run.log_artifact(art); print("Documentation run:", run.url)
